# Balance-law repair projection

This notebook illustrates the projected correction coordinates used as the first step toward balance-law repair. The example uses synthetic modes and geometric time units `t/M`. It does not minimize a balance-law residual yet; instead it shows how raw fractional mode corrections are projected away from protected alignment directions before being applied.

In [ ]:
from __future__ import annotations

import matplotlib.pyplot as plt
import numpy as np

from waveformtools.bms_frame_diagnostics import compute_bms_frame_diagnostics
from waveformtools.bms_frame_preprocessing import preprocess_bms_frame
from waveformtools.balance_law_repair import (
    BalanceLawRepairSpec,
    apply_balance_law_repair_coefficients,
    build_balance_law_repair_projection,
)
from waveformtools.modes_array import ModesArray
from waveformtools.waveform_corrections import (
    FractionalModeCorrectionSpec,
    coefficient_index,
    zero_fractional_correction_vector,
)


## Synthetic waveform modes

In [ ]:
def make_synthetic_modes():
    t_over_M = np.linspace(-120.0, 80.0, 512)
    envelope = np.exp(-0.0007 * (t_over_M + 20.0) ** 2)
    phase = 0.075 * t_over_M + 0.00045 * t_over_M**2

    modes = ModesArray(ell_max=3, time_axis=t_over_M, spin_weight=-2)
    modes.create_modes_array(ell_max=3, data_len=len(t_over_M))
    modes.set_mode_data(ell=2, emm=2, data=envelope * np.exp(1j * phase))
    modes.set_mode_data(ell=2, emm=-2, data=0.35 * envelope * np.exp(-1j * phase))
    modes.set_mode_data(ell=3, emm=1, data=0.18 * envelope * np.exp(0.5j * phase))
    return modes


unrepaired_modes = make_synthetic_modes()


## BMS-frame diagnostics for comparison metadata

In [ ]:
frame_diagnostics = compute_bms_frame_diagnostics(
    unrepaired_modes,
    initial_mass=1.0,
    final_mass=0.95,
    reference_frame="synthetic input frame",
    compute_memory_finite_time=False,
)
print("energy radiated:", frame_diagnostics.energy_radiated)
print("radiated linear momentum:", frame_diagnostics.radiated_linear_momentum)
print("kick proxy:", frame_diagnostics.kick_velocity)
print("radiated angular momentum:", frame_diagnostics.angular_momentum_radiated)
frame_diagnostics.assumptions


## Conservative BMS-frame preprocessing report

In [ ]:
candidate_modes = make_synthetic_modes()
target_pre, candidate_pre, frame_report = preprocess_bms_frame(
    unrepaired_modes,
    candidate_modes,
    diagnostics={"initial_mass": 1.0, "final_mass": 0.95},
)
print("compatible:", frame_report.compatibility["compatible"])
print("failed checks:", frame_report.compatibility["failed_checks"])
frame_report.transforms


## Project raw corrections away from protected directions

In [ ]:
correction_spec = FractionalModeCorrectionSpec(
    modes=[(2, 2), (2, -2)],
    n_time_basis=7,
    max_abs_delta={(2, 2): 0.03, (2, -2): 0.03},
)
repair_spec = BalanceLawRepairSpec(
    correction=correction_spec,
    protect_time_shift=True,
    protect_global_phase=True,
    protect_orbital_phase=True,
)

projection = build_balance_law_repair_projection(unrepaired_modes, repair_spec)
projection.diagnostics


## Apply a small projected repair candidate

In [ ]:
coefficients = zero_fractional_correction_vector(correction_spec)
coefficients[coefficient_index(correction_spec, (2, 2), 2, "real")] = 0.018
coefficients[coefficient_index(correction_spec, (2, 2), 4, "imag")] = -0.012
coefficients[coefficient_index(correction_spec, (2, -2), 3, "real")] = -0.01

repair_result = apply_balance_law_repair_coefficients(
    unrepaired_modes,
    coefficients,
    projection,
)
repaired_modes = repair_result.corrected_modes
repair_result.diagnostics


## Plot unrepaired and repaired modes

In [ ]:
mode = (2, 2)
t_over_M = unrepaired_modes.time_axis
h_unrepaired = unrepaired_modes.mode(*mode)
h_repaired = repaired_modes.mode(*mode)
delta_h = h_repaired - h_unrepaired
fractional_delta = delta_h / np.maximum(np.abs(h_unrepaired), 1e-12)

mode_limit = 1.1 * max(
    np.max(np.abs(np.real(h_unrepaired))),
    np.max(np.abs(np.imag(h_unrepaired))),
    np.max(np.abs(np.real(h_repaired))),
    np.max(np.abs(np.imag(h_repaired))),
)
difference_limit = 1.1 * max(np.max(np.abs(np.real(delta_h))), np.max(np.abs(np.imag(delta_h))), 1e-6)
fraction_limit = 1.1 * max(np.max(np.abs(np.real(fractional_delta))), np.max(np.abs(np.imag(fractional_delta))), 1e-4)

fig, axes = plt.subplots(3, 1, figsize=(8, 9), sharex=True)
axes[0].plot(t_over_M, np.real(h_unrepaired), label="Re unrepaired")
axes[0].plot(t_over_M, np.imag(h_unrepaired), label="Im unrepaired")
axes[0].plot(t_over_M, np.real(h_repaired), "--", label="Re repaired")
axes[0].plot(t_over_M, np.imag(h_repaired), "--", label="Im repaired")
axes[0].set_ylabel(r"$h_{22}$")
axes[0].set_ylim(-mode_limit, mode_limit)
axes[0].legend(ncol=2)

axes[1].plot(t_over_M, np.real(delta_h), label=r"Re $\Delta h_{22}$")
axes[1].plot(t_over_M, np.imag(delta_h), label=r"Im $\Delta h_{22}$")
axes[1].set_ylabel(r"$\Delta h_{22}$")
axes[1].set_ylim(-difference_limit, difference_limit)
axes[1].legend(ncol=2)

axes[2].plot(t_over_M, np.real(fractional_delta), label=r"Re $\Delta h_{22}/|h_{22}|$")
axes[2].plot(t_over_M, np.imag(fractional_delta), label=r"Im $\Delta h_{22}/|h_{22}|$")
axes[2].set_ylabel("fractional change")
axes[2].set_xlabel(r"$t/M$")
axes[2].set_ylim(-fraction_limit, fraction_limit)
axes[2].legend(ncol=2)

fig.suptitle("Projected balance-law repair candidate for mode (2, 2)")
fig.tight_layout()
